In [0]:
#Call libraries
from pyspark.sql import functions as F

In [0]:
#Read bronze table
bronze_table = "workspace.sudoku_bronze.raw_games"
bronze_df = spark.table(bronze_table)
print(f"Total Bronze records: {bronze_df.count():,}")

In [0]:
#Check for null values ​​and the length of the boards (81 is the maximum number of cells a game can have)
#Evaluate structural data quality
quality_df = bronze_df.select(
    F.count("*").alias("total_records"),
    F.sum(
        F.when(
            F.col("quizzes").isNull() | F.col("solutions").isNull(),
            1
        ).otherwise(0)
    ).alias("null_records"),
    F.sum(
        F.when(
            (F.length("quizzes") != 81) |
            (F.length("solutions") != 81),
            1
        ).otherwise(0)
    ).alias("invalid_length_records"),
    F.sum(
        F.when(
            (~F.col("quizzes").rlike("^[0-9]{81}$")) |
            (~F.col("solutions").rlike("^[1-9]{81}$")),
            1
        ).otherwise(0)
    ).alias("invalid_character_records")
)

display(quality_df)

In [0]:
#Count duplicated Sudoku puzzles
duplicate_df = (
    bronze_df
    .groupBy("quizzes")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_groups = duplicate_df.count()
print(f"Duplicated puzzle groups: {duplicate_groups:,}")
display(duplicate_df.limit(10))

In [0]:
#The numbers already filled in the challenge must correspond to the same numbers in the solution
#Compare each given digit with the corresponding solution digit
mismatch_condition = F.expr("""
    exists(
        sequence(1, 81),
        i -> substring(quizzes, i, 1) <> '0'
             AND substring(quizzes, i, 1) <> substring(solutions, i, 1)
    )
""")

consistency_df = bronze_df.select(
    F.count("*").alias("total_records"),
    F.sum(
        F.when(mismatch_condition, 1).otherwise(0)
    ).alias("inconsistent_solutions")
)

display(consistency_df)

In [0]:
#Before creating the Silver table, we need to verify that each solution adheres to three rules:
    #1. Each row contains the numbers 1 through 9, with no repetitions -> j
    #2. Each column contains the numbers 1 through 9, with no repetitions -> i
    #3. Each 3×3 block contains the numbers 1 through 9, with no repetitions -> b (block) & k (position in the block)

#Convert each solution into an array of 81 digits
solutions_df = bronze_df.withColumn(
    "solution_digits",
    F.split(F.col("solutions"), "")
)

#Validate all rows, columns, and 3x3 boxes
#Executes a Spark SQL expression within PySpark
validation_df = solutions_df.withColumn(
    "valid_solution",
    F.expr("""
        forall(sequence(0, 8), i ->
            size(array_distinct(slice(solution_digits, i * 9 + 1, 9))) = 9
        )
        AND
        forall(sequence(0, 8), i ->
            size(array_distinct(transform(sequence(0, 8),
                j -> element_at(solution_digits, j * 9 + i + 1))
            )) = 9
        )
        AND
        forall(sequence(0, 8), b ->
            size(array_distinct(transform(sequence(0, 8),
                k -> element_at(solution_digits,
                    CAST((b DIV 3 * 3 + k DIV 3) * 9 + (b % 3 * 3 + k % 3) + 1 AS INT)))
            )) = 9
        )
    """)
)


#Summarize mathematical validation results
validation_df.groupBy("valid_solution").count().show()

In [0]:
#Prepare validated records and calculate basic features
silver_df = (
    validation_df
    .filter(F.col("valid_solution") == True)
    .select(
        "quizzes",
        "solutions",
        "ingestion_timestamp",
        "source_file",
        "valid_solution",
        F.length(
            F.regexp_replace("quizzes", "0", "")
        ).alias("given_cells"), #Number of cells originally filled in
        (
            F.length("quizzes")
            - F.length(F.regexp_replace("quizzes", "0", ""))
        ).alias("empty_cells") #Number of empty positions, represented by zero
    )
)

silver_df.printSchema()
display(silver_df.limit(5))

In [0]:
#Define the Silver table
silver_table = "workspace.sudoku_silver.validated_games"

#Persist validated records as a Delta table
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

print(f"Silver table created successfully: {silver_table}")

In [0]:
#Read the persisted Silver table
silver_table_df = spark.table(silver_table)

#Validate record count
print(f"Silver records: {silver_table_df.count():,}")

#Calculate basic puzzle statistics
silver_table_df.select(
    F.min("given_cells").alias("min_given_cells"),
    F.max("given_cells").alias("max_given_cells"),
    F.avg("given_cells").alias("avg_given_cells"),
    F.min("empty_cells").alias("min_empty_cells"),
    F.max("empty_cells").alias("max_empty_cells"),
    F.avg("empty_cells").alias("avg_empty_cells")
).show()

In [0]:
#Check whether each puzzle contains exactly 81 cells
invalid_cell_counts = silver_table_df.filter(
    (F.col("given_cells") + F.col("empty_cells")) != 81
).count()

print(f"Invalid cell counts: {invalid_cell_counts:,}")